## Get EXIF Data From Videos

In [1]:
# Import necessary libraries
import os
import subprocess
import pandas as pd
import logging
from glob import glob
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from datetime import datetime

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger('exif_extractor')

In [2]:
def extract_exif_to_df(files=None, directory=".", file_pattern="*.mp4", output_dir=None, 
                      extract_gps=True, log_level=logging.INFO):
    """
    Extracts EXIF metadata including time-series GPS data from GoPro videos.
    
    Args:
        files (list): Optional specific files to process.
        directory (str): Directory to search for files if files not specified.
        file_pattern (str): Pattern to match files (e.g., '*.mp4').
        output_dir (str): Directory for output files (defaults to input directory).
        extract_gps (bool): Whether to extract and parse GPS data specifically.
        log_level (int): Logging level.
        
    Returns:
        pd.DataFrame: DataFrame with extracted metadata.
        pd.DataFrame: DataFrame with GPS data if extract_gps is True.
    """
    logger.setLevel(log_level)
    
    # Set output directory
    if output_dir is None:
        output_dir = directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Get files to process
    if files:
        # Use provided files list
        file_list = files
        logger.info(f"Processing {len(file_list)} specified files")
    else:
        # Handle case-insensitive file matching
        if file_pattern.startswith('*.'):
            extension = file_pattern[2:]
            file_list = []
            for ext in [extension.lower(), extension.upper()]:
                file_list.extend(glob(os.path.join(directory, f"*.{ext}")))
        else:
            file_list = glob(os.path.join(directory, file_pattern))
        
        logger.info(f"Found {len(file_list)} files matching pattern '{file_pattern}' in '{directory}'")
    
    if not file_list:
        logger.warning(f"No files found to process")
        return pd.DataFrame(), pd.DataFrame() if extract_gps else None
    
    all_rows = []
    gps_data_rows = []
    
    for file in file_list:
        logger.info(f"Processing file: {file}")
        file_info = {
            "filename": os.path.basename(file),
            "filepath": os.path.abspath(file),
            "filesize_bytes": os.path.getsize(file),
            "filesize_mb": os.path.getsize(file) / (1024 * 1024),
            "creation_time": pd.Timestamp(os.stat(file).st_ctime, unit='s'),
            "modification_time": pd.Timestamp(os.stat(file).st_mtime, unit='s'),
            "has_exif": True
        }
        
        try:
            # Run exiftool with enhanced GPS extraction
            cmd = ["exiftool", "-G", "-ee", "-api", "LargeFileSupport=1", file]
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
            
            # Save raw output to text file for inspection
            output_file = os.path.join(output_dir, f"{os.path.splitext(os.path.basename(file))[0]}_exif.txt")
            with open(output_file, 'w') as f:
                f.write(result.stdout)
            logger.info(f"Saved raw EXIF data to {output_file}")
            
            # Parse the basic metadata
            row = file_info.copy()
            metadata = {}
            
            # For GPS data extraction
            current_gps_record = None
            gps_records = []
            
            lines = result.stdout.strip().split("\n")
            for line in lines:
                if ": " not in line:
                    continue
                
                tag_part, value = line.split(": ", 1)
                tag = tag_part.strip()
                
                # Clean tag name - extract section as prefix
                section = None
                for prefix in ["[GoPro]", "[QuickTime]", "[File]", "[Composite]", "[ExifTool]"]:
                    if tag.startswith(prefix):
                        section = prefix.strip("[]")
                        tag = tag[len(prefix):].strip()
                        break
                
                # If no section found, use "Other"
                if section is None:
                    section = "Other"
                
                # Skip binary data entries
                if "(Binary data" in value:
                    continue
                
                # Add section prefix to avoid column name collisions
                column_name = f"{section}_{tag}"
                
                # Extract GPS data into separate structure
                if extract_gps and section == "GoPro" and "GPS" in tag:
                    # Handle GPS data
                    if tag == "GPS Latitude" and value:
                        # Start new GPS record if needed
                        if current_gps_record is None or "latitude" in current_gps_record:
                            current_gps_record = {"filename": file_info["filename"]}
                        current_gps_record["latitude"] = value
                    elif tag == "GPS Longitude" and value:
                        if current_gps_record is not None:
                            current_gps_record["longitude"] = value
                    elif tag == "GPS Altitude" and value:
                        if current_gps_record is not None:
                            current_gps_record["altitude"] = value.split()[0]  # Strip units
                    elif tag == "GPS Speed" and value:
                        if current_gps_record is not None:
                            current_gps_record["speed"] = value
                    elif tag == "GPS Date Time" and value:
                        if current_gps_record is not None:
                            current_gps_record["timestamp"] = value
                            # Save complete GPS record and start fresh
                            if all(k in current_gps_record for k in ["latitude", "longitude", "timestamp"]):
                                gps_records.append(current_gps_record.copy())
                                current_gps_record = None
                
                # Add to metadata dictionary
                metadata[column_name] = value
            
            # Add all metadata to row
            row.update(metadata)
            all_rows.append(row)
            
            # Process GPS data into the DataFrame
            if extract_gps and gps_records:
                for record in gps_records:
                    gps_data_rows.append(record)
            
        except subprocess.TimeoutExpired:
            logger.error(f"Timeout while processing '{file}'")
            file_info["has_exif"] = False
            all_rows.append(file_info)
        except Exception as e:
            logger.error(f"Error processing '{file}': {str(e)}")
            file_info["has_exif"] = False
            all_rows.append(file_info)
    
    # Create metadata DataFrame
    if all_rows:
        result_df = pd.DataFrame(all_rows)
        logger.info(f"Processed {len(file_list)} files. Created {len(result_df)} rows.")
        logger.info(f"Raw EXIF data saved to '{output_dir}' directory with '_exif.txt' suffix")
    else:
        logger.warning("No metadata could be extracted from any files.")
        result_df = pd.DataFrame()
    
    # Create GPS DataFrame
    if extract_gps:
        if gps_data_rows:
            gps_df = pd.DataFrame(gps_data_rows)
            logger.info(f"Extracted {len(gps_df)} GPS data points.")
            # Save GPS data to CSV
            gps_csv = os.path.join(output_dir, "gps_data.csv")
            gps_df.to_csv(gps_csv, index=False)
            logger.info(f"GPS data saved to {gps_csv}")
            return result_df, gps_df
        else:
            logger.warning("No GPS data found in the processed files.")
            return result_df, pd.DataFrame()
    else:
        return result_df, None

# Function to visualize GPS data
def visualize_gps_data(gps_df, output_dir="."):
    """
    Creates visualizations for GPS data.
    
    Args:
        gps_df (pd.DataFrame): DataFrame with GPS data.
        output_dir (str): Directory to save visualizations.
    """
    if gps_df.empty:
        logger.warning("No GPS data to visualize")
        return
    
    # Convert coordinates to decimal degrees for plotting
    def parse_coordinates(coord_str):
        if pd.isna(coord_str) or not coord_str:
            return None
        
        try:
            # Parse format like "19 deg 3' 24.16\" N"
            parts = coord_str.split()
            degrees = float(parts[0])
            minutes = float(parts[2].strip("'"))
            seconds = float(parts[3].strip('"'))
            direction = parts[4]
            
            decimal = degrees + minutes/60 + seconds/3600
            if direction in ['S', 'W']:
                decimal = -decimal
            
            return decimal
        except Exception as e:
            logger.error(f"Error parsing coordinate: {coord_str} - {str(e)}")
            return None
    
    # Add decimal coordinates
    gps_df['lat_decimal'] = gps_df['latitude'].apply(parse_coordinates)
    gps_df['lon_decimal'] = gps_df['longitude'].apply(parse_coordinates)
    
    # Convert timestamps to datetime
    gps_df['datetime'] = pd.to_datetime(gps_df['timestamp'], errors='coerce')
    
    # Drop rows with invalid coordinates
    gps_df = gps_df.dropna(subset=['lat_decimal', 'lon_decimal'])
    
    if gps_df.empty:
        logger.warning("No valid GPS coordinates to visualize")
        return
    
    # Create a simple scatter plot of coordinates
    plt.figure(figsize=(10, 6))
    for filename in gps_df['filename'].unique():
        file_data = gps_df[gps_df['filename'] == filename]
        plt.plot(file_data['lon_decimal'], file_data['lat_decimal'], 'o-', label=filename)
    
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title('GPS Coordinates from GoPro Videos')
    plt.legend()
    plt.grid(True)
    
    # Save the plot
    plot_path = os.path.join(output_dir, "gps_plot.png")
    plt.savefig(plot_path)
    plt.close()
    
    logger.info(f"GPS visualization saved to {plot_path}")
    
    # Display statistics
    print(f"GPS Summary Statistics:")
    print(f"- Total points: {len(gps_df)}")
    print(f"- Files: {', '.join(gps_df['filename'].unique())}")
    print(f"- Coordinate range: Lat [{gps_df['lat_decimal'].min():.6f}, {gps_df['lat_decimal'].max():.6f}], "
          f"Lon [{gps_df['lon_decimal'].min():.6f}, {gps_df['lon_decimal'].max():.6f}]")
    
    return plot_path

In [3]:
# Set parameters for your extraction
VIDEO_DIR = "../vids/"  # Change this to your directory with the 3 video files
OUTPUT_DIR = "../exif_out/"  # Where to save the output files
FILE_PATTERN = "*.mp4"  # Pattern to match your files (or specify exact files below)


# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Run the extraction
metadata_df, gps_df = extract_exif_to_df(
    # files=SPECIFIC_FILES,  # Uncomment to use specific files
    directory=VIDEO_DIR,
    file_pattern=FILE_PATTERN,
    output_dir=OUTPUT_DIR,
    extract_gps=True,
    log_level=logging.INFO
)

# Display summary of extracted data
print(f"Extracted metadata from {len(metadata_df)} files")
print(f"Found {len(gps_df) if gps_df is not None else 0} GPS data points")

# Display the files that were processed
display(metadata_df[['filename', 'filesize_mb', 'has_exif']].sort_values('filename'))

2025-03-24 21:25:25 - INFO - Found 9 files matching pattern '*.mp4' in '../vids/'
2025-03-24 21:25:25 - INFO - Processing file: ../vids/test_drive_5.mp4
2025-03-24 21:25:26 - INFO - Saved raw EXIF data to ../exif_out/test_drive_5_exif.txt
2025-03-24 21:25:26 - INFO - Processing file: ../vids/test_drive_6.mp4
2025-03-24 21:25:26 - INFO - Saved raw EXIF data to ../exif_out/test_drive_6_exif.txt
2025-03-24 21:25:26 - INFO - Processing file: ../vids/test_drive_8.mp4
2025-03-24 21:25:27 - INFO - Saved raw EXIF data to ../exif_out/test_drive_8_exif.txt
2025-03-24 21:25:27 - INFO - Processing file: ../vids/test_drive_7.MP4
2025-03-24 21:25:31 - INFO - Saved raw EXIF data to ../exif_out/test_drive_7_exif.txt
2025-03-24 21:25:31 - INFO - Processing file: ../vids/Test_drive_3.MP4
2025-03-24 21:25:40 - INFO - Saved raw EXIF data to ../exif_out/Test_drive_3_exif.txt
2025-03-24 21:25:40 - INFO - Processing file: ../vids/test_drive_4.MP4
2025-03-24 21:25:45 - INFO - Saved raw EXIF data to ../exif_ou

Extracted metadata from 9 files
Found 16218 GPS data points


,filename,filesize_mb,has_exif
4,Test_drive_3.MP4,3838.372286,True
6,gps_test_1.MP4,110.707814,True
7,gps_test_2.MP4,111.099422,True
8,gps_test_3.MP4,100.669154,True
5,test_drive_4.MP4,2504.109884,True
0,test_drive_5.mp4,1448.039396,True
1,test_drive_6.mp4,1877.096504,True
3,test_drive_7.MP4,2058.901525,True
2,test_drive_8.mp4,2623.881537,True


In [47]:
gps_df

,filename,latitude,longitude,altitude,speed,timestamp
0,test_drive_7.MP4,"19 deg 4' 18.74"" N","72 deg 59' 55.23"" E",10.86,0,2025:03:23 06:27:19.700
1,test_drive_7.MP4,"19 deg 4' 18.74"" N","72 deg 59' 55.23"" E",10.86,0,2025:03:23 06:27:19.800
2,test_drive_7.MP4,"19 deg 4' 18.74"" N","72 deg 59' 55.23"" E",10.86,0,2025:03:23 06:27:19.900
3,test_drive_7.MP4,"19 deg 4' 18.74"" N","72 deg 59' 55.23"" E",10.86,0,2025:03:23 06:27:20.000
4,test_drive_7.MP4,"19 deg 4' 18.74"" N","72 deg 59' 55.23"" E",10.86,0,2025:03:23 06:27:20.100
...,...,...,...,...,...,...
15592,test_drive_4.MP4,"19 deg 5' 30.03"" N","73 deg 0' 42.94"" E",-5.218,0.8208,2025:03:21 05:57:31.199
15593,test_drive_4.MP4,"19 deg 5' 30.03"" N","73 deg 0' 42.94"" E",-5.187,0.2808,2025:03:21 05:57:31.299
15594,test_drive_4.MP4,"19 deg 5' 30.03"" N","73 deg 0' 42.94"" E",-5.179,0.5076,2025:03:21 05:57:31.399
15595,test_drive_4.MP4,"19 deg 5' 30.03"" N","73 deg 0' 42.94"" E",-5.163,0.7776,2025:03:21 05:57:31.499
